In [1]:
from pathlib import Path

files = sorted(Path(r"C:\Users\bhuwa\Desktop\dell-pipeline\data").rglob("*.csv"))
print("CSV files found:", len(files))
print(files[0])
print(files[-1])

CSV files found: 91
C:\Users\bhuwa\Desktop\dell-pipeline\data\data_Q1_2024\2024-01-01.csv
C:\Users\bhuwa\Desktop\dell-pipeline\data\data_Q1_2024\2024-03-31.csv


In [2]:
import pandas as pd

df = pd.read_csv(files[0])
print(df.shape)
df.head()

(274660, 193)


,date,serial_number,model,capacity_bytes,failure,datacenter,cluster_id,vault_id,pod_id,pod_slot_num,...,smart_250_normalized,smart_250_raw,smart_251_normalized,smart_251_raw,smart_252_normalized,smart_252_raw,smart_254_normalized,smart_254_raw,smart_255_normalized,smart_255_raw
0,2024-01-01,WD-WX31DB48X22V,WDC WD60EFRX,6001175126016,0,sac0,0,1002,0,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-01-01,Z4D00WGP,ST6000DX000,6001175126016,0,sac0,0,1002,0,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-01-01,Z4D00YC6,ST6000DX000,6001175126016,0,sac0,0,1002,0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-01-01,Z4D00YS2,ST6000DX000,6001175126016,0,sac0,0,1002,0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-01-01,Z4D029JS,ST6000DX000,6001175126016,0,sac0,0,1002,0,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
print("Failures this day:", df["failure"].sum())
print("Drives this day:", df["serial_number"].nunique())

Failures this day: 3
Drives this day: 274660


In [4]:
missing = df.isna().mean().sort_values()
print(missing.head(25))

date                    0.000000
serial_number           0.000000
model                   0.000000
capacity_bytes          0.000000
failure                 0.000000
cluster_id              0.000000
vault_id                0.000000
pod_id                  0.000000
is_legacy_format        0.000000
smart_9_normalized      0.001569
smart_12_normalized     0.001569
smart_9_raw             0.001569
smart_12_raw            0.001569
smart_194_normalized    0.001569
smart_194_raw           0.001569
smart_1_raw             0.002487
smart_1_normalized      0.002487
datacenter              0.004442
smart_192_normalized    0.005086
smart_192_raw           0.005086
smart_5_raw             0.008494
smart_5_normalized      0.008494
smart_199_raw           0.008494
smart_199_normalized    0.008494
smart_198_normalized    0.009412
dtype: float64


In [5]:
good_cols = missing[missing < 0.5].index.tolist()
print(len(good_cols))
print(good_cols)

47
['date', 'serial_number', 'model', 'capacity_bytes', 'failure', 'cluster_id', 'vault_id', 'pod_id', 'is_legacy_format', 'smart_9_normalized', 'smart_12_normalized', 'smart_9_raw', 'smart_12_raw', 'smart_194_normalized', 'smart_194_raw', 'smart_1_raw', 'smart_1_normalized', 'datacenter', 'smart_192_normalized', 'smart_192_raw', 'smart_5_raw', 'smart_5_normalized', 'smart_199_raw', 'smart_199_normalized', 'smart_198_normalized', 'smart_198_raw', 'smart_10_normalized', 'smart_7_raw', 'smart_10_raw', 'smart_3_raw', 'smart_4_normalized', 'smart_7_normalized', 'smart_3_normalized', 'smart_4_raw', 'smart_193_raw', 'smart_193_normalized', 'pod_slot_num', 'smart_197_normalized', 'smart_197_raw', 'smart_240_normalized', 'smart_240_raw', 'smart_196_normalized', 'smart_196_raw', 'smart_8_normalized', 'smart_2_normalized', 'smart_8_raw', 'smart_2_raw']


In [6]:
smart_cols = ["smart_5_raw", "smart_9_raw", "smart_194_raw",
              "smart_197_raw", "smart_198_raw"]

print(df[smart_cols].isna().mean())
print(df[smart_cols].describe())

smart_5_raw      0.008494
smart_9_raw      0.001569
smart_194_raw    0.001569
smart_197_raw    0.031257
smart_198_raw    0.009412
dtype: float64
         smart_5_raw    smart_9_raw  smart_194_raw  smart_197_raw  \
count  272327.000000  274229.000000  274229.000000  266075.000000   
mean       28.458093   28413.472171      32.565593       0.950894   
std       829.284703   19521.427338       5.263244     163.862740   
min         0.000000       1.000000      15.000000       0.000000   
25%         0.000000   13155.000000      29.000000       0.000000   
50%         0.000000   25204.000000      33.000000       0.000000   
75%         0.000000   36340.000000      36.000000       0.000000   
max     65440.000000   93588.000000      61.000000   81064.000000   

       smart_198_raw  
count  272075.000000  
mean        0.762360  
std       160.955602  
min         0.000000  
25%         0.000000  
50%         0.000000  
75%         0.000000  
max     81064.000000  


In [7]:
cols = ["date", "serial_number", "failure"] + smart_cols

frames = []
for f in files:
    day = pd.read_csv(f, usecols=cols)
    frames.append(day)

full = pd.concat(frames, ignore_index=True)
print(full.shape)
print("Total failures:", full["failure"].sum())

(25189213, 8)
Total failures: 978


In [8]:
bad = ["smart_5_raw", "smart_197_raw", "smart_198_raw"]

print((full[bad] > 0).groupby(full["failure"]).mean())
print(full.groupby("failure")[["smart_9_raw", "smart_194_raw"]].mean())

         smart_5_raw  smart_197_raw  smart_198_raw
failure                                           
0           0.035267        0.01157       0.007873
1           0.575665        0.49591       0.348671
          smart_9_raw  smart_194_raw
failure                             
0        28356.270573      32.601580
1        34988.660843      32.879753


In [9]:
%pip install pyarrow

   ---------------------------------------- 0.0/28.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/28.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/28.5 MB ? eta -:--:--
    --------------------------------------- 0.5/28.5 MB 1.5 MB/s eta 0:00:19
   - -------------------------------------- 0.8/28.5 MB 1.5 MB/s eta 0:00:19
   - -------------------------------------- 1.0/28.5 MB 1.3 MB/s eta 0:00:21
   - -------------------------------------- 1.3/28.5 MB 1.2 MB/s eta 0:00:23
   - -------------------------------------- 1.3/28.5 MB 1.2 MB/s eta 0:00:23
   -- ------------------------------------- 1.6/28.5 MB 1.2 MB/s eta 0:00:23
   -- ------------------------------------- 1.8/28.5 MB 1.2 MB/s eta 0:00:23
   -- ------------------------------------- 2.1/28.5 MB 1.1 MB/s eta 0:00:24
   -- ------------------------------------- 2.1/28.5 MB 1.1 MB/s eta 0:00:24
   -- ------------------------------------- 2.1/28.5 MB 1.1 MB/s eta 0:00:24
   --- -------------


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import sys
print(sys.executable)

import pyarrow
print("pyarrow version:", pyarrow.__version__)

c:\Users\bhuwa\Desktop\dell-pipeline\venv\Scripts\python.exe
pyarrow version: 26.0.0


In [1]:
import pandas as pd
from pathlib import Path

data_dir = Path(r"C:\Users\bhuwa\Desktop\dell-pipeline\data")
files = sorted(data_dir.rglob("*.csv"))
print("CSV files found:", len(files))

smart_cols = ["smart_5_raw", "smart_9_raw", "smart_194_raw",
              "smart_197_raw", "smart_198_raw"]
cols = ["date", "serial_number", "failure"] + smart_cols

frames = []
for f in files:
    frames.append(pd.read_csv(f, usecols=cols))
    print("loaded", f.name)

full = pd.concat(frames, ignore_index=True)
print(full.shape)
print("Total failures:", full["failure"].sum())

out_path = data_dir / "full_q1_2024.parquet"
full.to_parquet(out_path, index=False)
print("saved:", out_path.exists(), round(out_path.stat().st_size / 1e6, 1), "MB")

CSV files found: 91
loaded 2024-01-01.csv
loaded 2024-01-02.csv
loaded 2024-01-03.csv
loaded 2024-01-04.csv
loaded 2024-01-05.csv
loaded 2024-01-06.csv
loaded 2024-01-07.csv
loaded 2024-01-08.csv
loaded 2024-01-09.csv
loaded 2024-01-10.csv
loaded 2024-01-11.csv
loaded 2024-01-12.csv
loaded 2024-01-13.csv
loaded 2024-01-14.csv
loaded 2024-01-15.csv
loaded 2024-01-16.csv
loaded 2024-01-17.csv
loaded 2024-01-18.csv
loaded 2024-01-19.csv
loaded 2024-01-20.csv
loaded 2024-01-21.csv
loaded 2024-01-22.csv
loaded 2024-01-23.csv
loaded 2024-01-24.csv
loaded 2024-01-25.csv
loaded 2024-01-26.csv
loaded 2024-01-27.csv
loaded 2024-01-28.csv
loaded 2024-01-29.csv
loaded 2024-01-30.csv
loaded 2024-01-31.csv
loaded 2024-02-01.csv
loaded 2024-02-02.csv
loaded 2024-02-03.csv
loaded 2024-02-04.csv
loaded 2024-02-05.csv
loaded 2024-02-06.csv
loaded 2024-02-07.csv
loaded 2024-02-08.csv
loaded 2024-02-09.csv
loaded 2024-02-10.csv
loaded 2024-02-11.csv
loaded 2024-02-12.csv
loaded 2024-02-13.csv
loaded 2024-

In [2]:
import pandas as pd

full = pd.read_parquet(r"C:\Users\bhuwa\Desktop\dell-pipeline\data\full_q1_2024.parquet")

print(full.shape)
print("Total failures:", full["failure"].sum())
print(full.dtypes)

(25189213, 8)
Total failures: 978
date                 str
serial_number        str
failure            int64
smart_5_raw      float64
smart_9_raw      float64
smart_194_raw    float64
smart_197_raw    float64
smart_198_raw    float64
dtype: object


In [3]:
full["date"] = pd.to_datetime(full["date"])

print(full["date"].min(), full["date"].max())
print("Unique drives:", full["serial_number"].nunique())

failed_serials = full.loc[full["failure"] == 1, "serial_number"]
print("Failed drives:", failed_serials.nunique())

2024-01-01 00:00:00 2024-03-31 00:00:00
Unique drives: 291331
Failed drives: 978


In [4]:
fail_date = full.loc[full["failure"] == 1].set_index("serial_number")["date"]

full["fail_date"] = full["serial_number"].map(fail_date)
full["days_to_failure"] = (full["fail_date"] - full["date"]).dt.days
full["label_7d"] = ((full["days_to_failure"] >= 0) & (full["days_to_failure"] <= 7)).astype("int8")

print(full["label_7d"].value_counts())

example = failed_serials.iloc[0]
print(full.loc[full["serial_number"] == example,
               ["date", "failure", "days_to_failure", "label_7d"]].tail(10))

label_7d
0    25181920
1        7293
Name: count, dtype: int64
            date  failure  days_to_failure  label_7d
61391 2024-01-01        1              0.0         1


In [5]:
late = full.loc[(full["failure"] == 1) & (full["date"] >= "2024-03-01"), "serial_number"].iloc[0]

print(full.loc[full["serial_number"] == late,
               ["date", "failure", "days_to_failure", "label_7d"]].tail(12))

               date  failure  days_to_failure  label_7d
13459641 2024-02-19        0             11.0         0
13735352 2024-02-20        0             10.0         0
14011007 2024-02-21        0              9.0         0
14287903 2024-02-22        0              8.0         0
14564795 2024-02-23        0              7.0         1
14841691 2024-02-24        0              6.0         1
15118590 2024-02-25        0              5.0         1
15395487 2024-02-26        0              4.0         1
15672388 2024-02-27        0              3.0         1
15949297 2024-02-28        0              2.0         1
16225278 2024-02-29        0              1.0         1
16502463 2024-03-01        1              0.0         1


In [6]:
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, precision_score

y_true = full["label_7d"]
y_pred = np.zeros(len(y_true), dtype="int8")

print("accuracy :", accuracy_score(y_true, y_pred))
print("recall   :", recall_score(y_true, y_pred))
print("precision:", precision_score(y_true, y_pred, zero_division=0))

accuracy : 0.9997104713037283
recall   : 0.0
precision: 0.0


In [7]:
train_end  = pd.Timestamp("2024-02-22")
test_start = pd.Timestamp("2024-03-01")
test_end   = pd.Timestamp("2024-03-24")

is_train = full["date"] <= train_end
is_test  = (full["date"] >= test_start) & (full["date"] <= test_end)

for name, mask in [("train", is_train), ("test", is_test)]:
    rows = mask.sum()
    pos = full.loc[mask, "label_7d"].sum()
    print(name, "rows:", rows, "| positives:", pos, "| rate:", round(pos / rows * 100, 4), "%")

print("rows dropped:", (~(is_train | is_test)).sum())

train rows: 14537206 | positives: 4862 | rate: 0.0334 %
test rows: 6728969 | positives: 1636 | rate: 0.0243 %
rows dropped: 3923038


In [8]:
X_train = full.loc[is_train, smart_cols].astype("float32")
y_train = full.loc[is_train, "label_7d"]

X_test = full.loc[is_test, smart_cols].astype("float32")
y_test = full.loc[is_test, "label_7d"]

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test :", X_test.shape, "| y_test :", y_test.shape)
print("positives -> train:", y_train.sum(), "| test:", y_test.sum())

X_train: (14537206, 5) | y_train: (14537206,)
X_test : (6728969, 5) | y_test : (6728969,)
positives -> train: 4862 | test: 1636


In [9]:
import time
from xgboost import XGBClassifier

n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
ratio = n_neg / n_pos
print("negatives:", n_neg, "| positives:", n_pos, "| ratio:", round(ratio, 1))

model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=ratio,
    tree_method="hist",
    eval_metric="aucpr",
    n_jobs=-1,
    random_state=42,
)

start = time.time()
model.fit(X_train, y_train)
print("training took", round(time.time() - start), "seconds")

negatives: 14532344 | positives: 4862 | ratio: 2989.0
training took 132 seconds


In [10]:
from sklearn.metrics import average_precision_score, confusion_matrix, precision_score, recall_score

proba = model.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, proba)
baseline = y_test.mean()
print("PR-AUC:", round(pr_auc, 4))
print("random-guess baseline:", round(baseline, 6))

pred = (proba >= 0.5).astype("int8")
print(confusion_matrix(y_test, pred))
print("recall   :", round(recall_score(y_test, pred), 4))
print("precision:", round(precision_score(y_test, pred, zero_division=0), 4))

PR-AUC: 0.0202
random-guess baseline: 0.000243
[[6367542  359791]
 [    645     991]]
recall   : 0.6057
precision: 0.0027


In [11]:
y_arr = y_test.to_numpy()

for t in [0.5, 0.7, 0.9, 0.95, 0.99, 0.999]:
    flagged = proba >= t
    n_flagged = flagged.sum()
    caught = (flagged & (y_arr == 1)).sum()
    recall = caught / y_arr.sum()
    precision = caught / n_flagged if n_flagged > 0 else 0
    print("cut-off", t, "| flagged:", n_flagged, "| caught:", caught,
          "| recall:", round(recall, 3), "| precision:", round(precision, 4))

cut-off 0.5 | flagged: 360782 | caught: 991 | recall: 0.606 | precision: 0.0027
cut-off 0.7 | flagged: 180759 | caught: 841 | recall: 0.514 | precision: 0.0047
cut-off 0.9 | flagged: 74774 | caught: 690 | recall: 0.422 | precision: 0.0092
cut-off 0.95 | flagged: 44861 | caught: 558 | recall: 0.341 | precision: 0.0124
cut-off 0.99 | flagged: 3487 | caught: 189 | recall: 0.116 | precision: 0.0542
cut-off 0.999 | flagged: 101 | caught: 13 | recall: 0.008 | precision: 0.1287


In [12]:
test_serials = full.loc[is_test, "serial_number"].reset_index(drop=True)

print("drives that really failed soon (test window):",
      test_serials[y_arr == 1].nunique())

for t in [0.5, 0.9, 0.99]:
    flagged = proba >= t
    flagged_drives = test_serials[flagged].nunique()
    caught_drives = test_serials[flagged & (y_arr == 1)].nunique()
    print("cut-off", t, "| rows flagged:", flagged.sum(),
          "| distinct drives flagged:", flagged_drives,
          "| failing drives caught:", caught_drives)

drives that really failed soon (test window): 272
cut-off 0.5 | rows flagged: 360782 | distinct drives flagged: 28535 | failing drives caught: 189
cut-off 0.9 | rows flagged: 74774 | distinct drives flagged: 4215 | failing drives caught: 146
cut-off 0.99 | rows flagged: 3487 | distinct drives flagged: 259 | failing drives caught: 42


In [13]:
imp = pd.Series(model.feature_importances_, index=smart_cols)
imp = imp.sort_values(ascending=False)
print(imp.round(4))

smart_5_raw      0.6560
smart_197_raw    0.2242
smart_9_raw      0.0535
smart_194_raw    0.0344
smart_198_raw    0.0318
dtype: float32


In [1]:
import sys
print(sys.executable)

import pandas as pd
import pyarrow
import mlflow
print("pyarrow:", pyarrow.__version__)
print("mlflow :", mlflow.__version__)

full = pd.read_parquet(r"C:\Users\bhuwa\Desktop\dell-pipeline\data\full_q1_2024.parquet")
print(full.shape)

c:\Users\bhuwa\Desktop\dell-pipeline\venv\Scripts\python.exe
pyarrow: 25.0.1
mlflow : 3.17.2
(25189213, 8)


In [2]:
smart_cols = ["smart_5_raw", "smart_9_raw", "smart_194_raw",
              "smart_197_raw", "smart_198_raw"]

# 1. real dates
full["date"] = pd.to_datetime(full["date"])

# 2. the "fails within 7 days" label
fail_date = full.loc[full["failure"] == 1].set_index("serial_number")["date"]
full["fail_date"] = full["serial_number"].map(fail_date)
full["days_to_failure"] = (full["fail_date"] - full["date"]).dt.days
full["label_7d"] = ((full["days_to_failure"] >= 0) & (full["days_to_failure"] <= 7)).astype("int8")

# 3. time-based split
train_end  = pd.Timestamp("2024-02-22")
test_start = pd.Timestamp("2024-03-01")
test_end   = pd.Timestamp("2024-03-24")
is_train = full["date"] <= train_end
is_test  = (full["date"] >= test_start) & (full["date"] <= test_end)

# 4. inputs (X) and answers (y)
X_train = full.loc[is_train, smart_cols].astype("float32")
y_train = full.loc[is_train, "label_7d"]
X_test  = full.loc[is_test, smart_cols].astype("float32")
y_test  = full.loc[is_test, "label_7d"]

print("label counts:", full["label_7d"].value_counts().to_dict())
print("train:", X_train.shape, "| positives:", y_train.sum())
print("test :", X_test.shape, "| positives:", y_test.sum())

label counts: {0: 25181920, 1: 7293}
train: (14537206, 5) | positives: 4862
test : (6728969, 5) | positives: 1636


In [3]:
import time
import mlflow
import mlflow.xgboost
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

project = Path(r"C:\Users\bhuwa\Desktop\dell-pipeline")
mlflow.set_tracking_uri("sqlite:///" + (project / "mlflow.db").as_posix())

exp_name = "drive-failure-7d"
if mlflow.get_experiment_by_name(exp_name) is None:
    mlflow.create_experiment(exp_name, artifact_location=(project / "mlartifacts").as_uri())
mlflow.set_experiment(exp_name)

n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())

params = {
    "n_estimators": 200,
    "max_depth": 4,
    "learning_rate": 0.1,
    "scale_pos_weight": n_neg / n_pos,
    "tree_method": "hist",
    "eval_metric": "aucpr",
    "n_jobs": -1,
    "random_state": 42,
}

with mlflow.start_run(run_name="baseline_xgb_200_trees"):
    mlflow.log_params(params)
    mlflow.log_params({
        "label_horizon_days": 7,
        "train_end": str(train_end.date()),
        "test_start": str(test_start.date()),
        "test_end": str(test_end.date()),
        "features": ",".join(smart_cols),
    })

    model = XGBClassifier(**params)
    start = time.time()
    model.fit(X_train, y_train)
    mlflow.log_metric("train_seconds", round(time.time() - start))

    proba = model.predict_proba(X_test)[:, 1]
    y_arr = y_test.to_numpy()
    mlflow.log_metric("pr_auc", average_precision_score(y_arr, proba))

    for tag, t in [("0_5", 0.5), ("0_99", 0.99)]:
        flagged = proba >= t
        caught = int((flagged & (y_arr == 1)).sum())
        mlflow.log_metric("recall_at_" + tag, caught / int(y_arr.sum()))
        mlflow.log_metric("precision_at_" + tag, caught / max(int(flagged.sum()), 1))
        mlflow.log_metric("rows_flagged_at_" + tag, int(flagged.sum()))

    mlflow.xgboost.log_model(model, name="model")
    print("run id:", mlflow.active_run().info.run_id)

print("done")

2026/10/24 03:00:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/24 03:00:11 INFO mlflow.store.db.utils: Updating database tables


run id: 0138eded657e42b596c62b09d7579614
done


In [4]:
import time
import mlflow
import mlflow.xgboost
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

cols_no_temp = ["smart_5_raw", "smart_9_raw", "smart_197_raw", "smart_198_raw"]

mlflow.set_experiment("drive-failure-7d")

with mlflow.start_run(run_name="no_temperature_xgb_200_trees"):
    mlflow.log_params(params)
    mlflow.log_params({
        "label_horizon_days": 7,
        "train_end": str(train_end.date()),
        "test_start": str(test_start.date()),
        "test_end": str(test_end.date()),
        "features": ",".join(cols_no_temp),
    })

    model_b = XGBClassifier(**params)
    start = time.time()
    model_b.fit(X_train[cols_no_temp], y_train)
    mlflow.log_metric("train_seconds", round(time.time() - start))

    proba_b = model_b.predict_proba(X_test[cols_no_temp])[:, 1]
    y_arr = y_test.to_numpy()
    pr_auc_b = average_precision_score(y_arr, proba_b)
    mlflow.log_metric("pr_auc", pr_auc_b)
    print("PR-AUC without temperature:", round(pr_auc_b, 4))

    for tag, t in [("0_5", 0.5), ("0_99", 0.99)]:
        flagged = proba_b >= t
        caught = int((flagged & (y_arr == 1)).sum())
        recall = caught / int(y_arr.sum())
        precision = caught / max(int(flagged.sum()), 1)
        mlflow.log_metric("recall_at_" + tag, recall)
        mlflow.log_metric("precision_at_" + tag, precision)
        mlflow.log_metric("rows_flagged_at_" + tag, int(flagged.sum()))
        print("cut-off", t, "| recall:", round(recall, 3), "| precision:", round(precision, 4))

    mlflow.xgboost.log_model(model_b, name="model")
    print("run id:", mlflow.active_run().info.run_id)

print("done")

PR-AUC without temperature: 0.0158
cut-off 0.5 | recall: 0.603 | precision: 0.0026
cut-off 0.99 | recall: 0.094 | precision: 0.0473
run id: 8aab2f0d7b7c4d1599afd10a5912148a
done
